In [ ]:
# Imports principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from skimage.io import imread
from skimage.transform import resize
from collections import Counter

# Modelos y métricas
from sklearn.model_selection import train_test_split, learning_curve, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn.pipeline import Pipeline

# --- CLASIFICADORES ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Para distribuciones de Random Search
from scipy.stats import randint, uniform

In [ ]:
# Celda 1: Función para cargar imágenes
def load_image_dataset(root_dir, size=(64,64), max_per_class=None):
    X, y = [], []
    classes = sorted(os.listdir(root_dir))
    for cls in classes:
        cls_path = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg'))]
        if max_per_class:
            files = files[:max_per_class]
        for f in files:
            img = imread(os.path.join(cls_path, f))
            if img.ndim == 3:  # convertir a escala de grises si es RGB
                img = img[...,0]
            img_resized = resize(img, size, anti_aliasing=True)
            X.append(img_resized)
            y.append(cls)
    return np.array(X), np.array(y), classes

# Celda 2: Carga, normalización y división de datos
# Cambia la ruta por la carpeta donde tengas las imágenes
X, y, classes = load_image_dataset("CMS_data", size=(64,64))
print("Datos cargados.")
print(f"Clases: {classes}")

# Normalizar al rango [0,1]
X = X / X.max()

# Aplanar imágenes para modelos clásicos
X_flat = X.reshape(len(X), -1)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Binarizar las etiquetas (necesario para ROC multiclase)
y_test_bin = label_binarize(y_test, classes=classes)
n_classes = y_test_bin.shape[1]
print(f"Número de clases binarizadas: {n_classes}")

In [ ]:
# --- Función para graficar ROC ---
def plot_multiclass_roc(y_test_bin, y_score, classes, title):
    plt.figure(figsize=(8,6))
    for i in range(len(classes)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{classes[i]} (AUC = {roc_auc:.2f})")
    plt.plot([0,1], [0,1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

# --- Función para graficar Curvas de Aprendizaje ---
def plot_learning_curve(estimator, title, X, y, ylim=None, cv=3, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)):
    plt.figure(figsize=(10, 6))
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.xlabel("Ejemplos de entrenamiento")
    plt.ylabel("Score (Accuracy)")
    
    train_sizes_abs, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes, scoring="accuracy"
    )
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    
    plt.grid(True)
    
    plt.fill_between(train_sizes_abs, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes_abs, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    
    plt.plot(train_sizes_abs, train_scores_mean, 'o-', color="r",
             label="Score de Entrenamiento (Train)")
    plt.plot(train_sizes_abs, test_scores_mean, 'o-', color="g",
             label="Score de Validación (Test/CV)")
    
    plt.legend(loc="best")
    plt.show()

In [ ]:
%%time
print("Iniciando RandomizedSearch para ANN (MLPClassifier)...")

# Es crucial escalar los datos para las Redes Neuronales
pipeline_ann = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(random_state=42, max_iter=1000, early_stopping=True, n_iter_no_change=10))
])

# Parámetros para el MLP (Red Neuronal)
param_dist_ann = {
    # Capas ocultas: 1 capa de 50 a 150 neuronas, o 2 capas
    'mlp__hidden_layer_sizes': [(50,), (100,), (150,), (50, 25), (100, 50)], 
    # Regularización L2 (distribución uniforme)
    'mlp__alpha': uniform(0.0001, 0.01),
    'mlp__activation': ['relu', 'tanh']
}

# n_iter=10: Probará 10 combinaciones aleatorias
random_ann = RandomizedSearchCV(
    pipeline_ann, 
    param_distributions=param_dist_ann, 
    n_iter=10, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1, 
    random_state=42, 
    verbose=1
)
random_ann.fit(X_train, y_train)

# Obtenemos el mejor pipeline (scaler + modelo)
best_ann = random_ann.best_estimator_
print(f"Mejores parámetros para ANN: {random_ann.best_params_}")

In [ ]:
print("--- Reporte de Clasificación (ANN) ---")
# El 'best_ann' (pipeline) se encarga de escalar X_test automáticamente
y_pred_ann = best_ann.predict(X_test)
print(classification_report(y_test, y_pred_ann, target_names=classes))

In [ ]:
print("--- Matriz de Confusión (ANN) ---")
cm_ann = confusion_matrix(y_test, y_pred_ann)
disp_ann = ConfusionMatrixDisplay(confusion_matrix=cm_ann, display_labels=classes)
disp_ann.plot(cmap="Reds", xticks_rotation=45)
plt.title("Matriz de Confusión - ANN (RandomSearch)")
plt.show()

In [ ]:
print("--- Precisión por Clase (ANN) ---")
report_ann_dict = classification_report(y_test, y_pred_ann, output_dict=True, target_names=classes)
df_ann = pd.DataFrame(report_ann_dict).transpose()

df_ann.loc[classes, "precision"].plot(kind="bar", color="red")
plt.title("Precisión por clase - ANN (RandomSearch)")
plt.ylabel("Precisión")
plt.ylim(0, 1.05); plt.xticks(rotation=45); plt.grid(axis='y', linestyle='--', alpha=0.7); plt.show()

In [ ]:
print("--- Curvas ROC (ANN) ---")
y_score_ann = best_ann.predict_proba(X_test)
plot_multiclass_roc(y_test_bin, y_score_ann, classes, "Curvas ROC - ANN (RandomSearch)")

In [ ]:
%%time
print("--- Curva de Aprendizaje (ANN) ---")
# Pasamos el pipeline completo 'best_ann' a la función
plot_learning_curve(best_ann, "Curva de Aprendizaje - ANN (RandomSearch)", X_train, y_train, cv=3, ylim=(0.4, 1.05))